# IMPORTACIÓN DE LIBRERÍAS

In [11]:
import pandas as pd
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10, Turbo256

output_notebook()

Loading BokehJS ...

# IMPORTACIÓN DE CSV
## Imprimimos el total de filas con shape
## Imprimimos las cabeceras de las columnas con columns.tolist
## Imprimimos las 10 primeras filas

In [12]:
df = pd.read_csv('data/deaths-in-armed-conflicts-by-region.csv')

print('Dimensiones del dataset:', df.shape)
print('\nColumnas:', df.columns.tolist())
print('\nPrimeras filas:')
df.head(10)

Dimensiones del dataset: (8056, 4)

Columnas: ['Entity', 'Code', 'Year', 'Deaths in armed conflicts']

Primeras filas:


,Entity,Code,Year,Deaths in armed conflicts
0,Abkhazia,OWID_ABK,1989,0
1,Abkhazia,OWID_ABK,1990,0
2,Abkhazia,OWID_ABK,1991,0
3,Abkhazia,OWID_ABK,1992,0
4,Abkhazia,OWID_ABK,1993,0
5,Abkhazia,OWID_ABK,1994,0
6,Abkhazia,OWID_ABK,1995,0
7,Abkhazia,OWID_ABK,1996,0
8,Abkhazia,OWID_ABK,1997,0
9,Abkhazia,OWID_ABK,1998,0


# MANEJO DE DATOS

In [13]:
# Primero identificamos los valores en cero para evitarlos en los siguientes cálculos para generar las siguientes columnas
df_nonzero = df[df['Deaths in armed conflicts'] > 0].copy()

# Identificamos o creamos la primera Columna: Año con más muertes por país.
idx_max = df_nonzero.groupby('Entity')['Deaths in armed conflicts'].idxmax()
year_max = df_nonzero.loc[idx_max, ['Entity', 'Year']].rename(
    columns={'Year': 'Año con más muertes'}
)

# Identificamos o creamos la segunda Columna: Año con menos muertes por país.
idx_min = df_nonzero.groupby('Entity')['Deaths in armed conflicts'].idxmin()
year_min = df_nonzero.loc[idx_min, ['Entity', 'Year']].rename(
    columns={'Year': 'Año con menos muertes'}
)

# Identificamos o creamos la tercera Columna: Promedio de muertes por país.
avg_deaths = (
    df_nonzero.groupby('Entity')['Deaths in armed conflicts']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'Deaths in armed conflicts': 'Promedio de muertes'})
)

# Identificamos o creamos la cuarta Columna: Media de muertes por país.
median_deaths = (
    df_nonzero.groupby('Entity')['Deaths in armed conflicts']
    .median()
    .reset_index()
    .rename(columns={'Deaths in armed conflicts': 'Mediana de muertes'})
)

# Añadimos las columnas creadas al dataset
summary = (
    year_max
    .merge(year_min,  on='Entity')
    .merge(avg_deaths, on='Entity')
    .merge(median_deaths, on='Entity')
)

summary = summary.reset_index(drop=True)

print(f'Países con al menos un año de conflicto registrado: {len(summary)}')
summary.head(15)

Países con al menos un año de conflicto registrado: 139


,Entity,Año con más muertes,Año con menos muertes,Promedio de muertes,Mediana de muertes
0,Abyei,2022,2009,61.60,47.0
1,Afghanistan,2021,2025,8378.18,5581.5
2,Africa,1994,2005,54654.50,20567.5
3,Albania,1999,1999,26.00,26.0
4,Algeria,1998,2020,724.17,339.0
5,Americas,2021,2026,6222.66,4560.5
6,Angola,1993,2022,1379.33,154.0
7,Argentina,1994,2025,63.50,63.5
8,Armenia,2022,2005,33.23,12.0
9,Asia and Oceania,2021,2026,15628.47,13910.0


# GRAFICOS

In [15]:
# Calculamos el total de muertes por país (excluyendo años con muertes de valor 0)
total_deaths = (
    df_nonzero.groupby('Entity')['Deaths in armed conflicts']
    .sum()
    .reset_index()
    .rename(columns={'Deaths in armed conflicts': 'Total muertes'})
)

# Combinamos con el resumen para tener año pico y promedio
top10 = (
    total_deaths
    .merge(summary[['Entity', 'Año con más muertes', 'Promedio de muertes']], on='Entity')
    .sort_values('Total muertes', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# Agregamos el valor de muertes en el peor año
peak_values = []
for _, row in top10.iterrows():
    val = df_nonzero[
        (df_nonzero['Entity'] == row['Entity']) &
        (df_nonzero['Year'] == row['Año con más muertes'])
    ]['Deaths in armed conflicts'].values[0]
    peak_values.append(val)

top10['Muertes año pico'] = peak_values

# Ordenamos para el gráfico
top10 = top10.sort_values('Total muertes', ascending=True).reset_index(drop=True)

countries = top10['Entity'].tolist()
palette = Turbo256[::26][:10]  # 10 colores distintos

source = ColumnDataSource(dict(
    countries=countries,
    total=top10['Total muertes'].tolist(),
    peak_year=top10['Año con más muertes'].astype(str).tolist(),
    peak_val=top10['Muertes año pico'].tolist(),
    promedio=top10['Promedio de muertes'].tolist(),
))

p1 = figure(
    y_range=countries,
    width=850,
    height=450,
    title='Top 10 países con más muertes acumuladas en conflictos armados',
    toolbar_location='right',
)

p1.hbar(
    y='countries',
    right='total',
    height=0.65,
    source=source,
    color=factor_cmap('countries', palette=palette, factors=countries),
    line_color='white',
    line_width=0.5,
)

hover1 = HoverTool(tooltips=[
    ('País',              '@countries'),
    ('Total muertes',     '@total{,}'),
    ('Año pico',          '@peak_year'),
    ('Muertes año pico',  '@peak_val{,}'),
    ('Promedio anual',    '@promedio{0,0.00}'),
])
p1.add_tools(hover1)

p1.xaxis.axis_label = 'Total de muertes acumuladas'
p1.yaxis.axis_label = 'País'
p1.xaxis.formatter.use_scientific = False
p1.title.text_font_size = '13px'
p1.axis.axis_label_text_font_style = 'bold'
p1.xgrid.grid_line_dash = [4, 4]
p1.ygrid.grid_line_color = None

show(p1)

In [18]:
# Top 5 países con MENOS muertes acumuladas (entre los que sí tuvieron conflictos, es decir, los que no tuvieron años con muertes en 0)
bottom5 = (
    total_deaths
    .merge(summary[['Entity', 'Año con menos muertes', 'Mediana de muertes']], on='Entity')
    .sort_values('Total muertes', ascending=True)
    .head(5)
    .reset_index(drop=True)
)

# Agregamos el valor de muertes en el mejor año (mínimo)
min_values = []
for _, row in bottom5.iterrows():
    val = df_nonzero[
        (df_nonzero['Entity'] == row['Entity']) &
        (df_nonzero['Year'] == row['Año con menos muertes'])
    ]['Deaths in armed conflicts'].values[0]
    min_values.append(val)

bottom5['Muertes año mín'] = min_values

# Ordenamos descendente para el gráfico horizontal
bottom5 = bottom5.sort_values('Total muertes', ascending=False).reset_index(drop=True)

countries5  = bottom5['Entity'].tolist()
palette5    = ['#595959', '#E8E8E8', '#BFBFBF', '#210124', '#DB504A']

source5 = ColumnDataSource(dict(
    countries=countries5,
    total=bottom5['Total muertes'].tolist(),
    min_year=bottom5['Año con menos muertes'].astype(str).tolist(),
    min_val=bottom5['Muertes año mín'].tolist(),
    mediana=bottom5['Mediana de muertes'].tolist(),
))

p2 = figure(
    y_range=countries5,
    width=850,
    height=350,
    title='Top 5 países con menos muertes acumuladas en conflictos armados (excluye años con 0)',
    toolbar_location='right',
)

p2.hbar(
    y='countries',
    right='total',
    height=0.55,
    source=source5,
    color=factor_cmap('countries', palette=palette5, factors=countries5),
    line_color='white',
    line_width=0.5,
)

hover2 = HoverTool(tooltips=[
    ('País',             '@countries'),
    ('Total muertes',    '@total{,}'),
    ('Año mínimo',       '@min_year'),
    ('Muertes ese año',  '@min_val{,}'),
    ('Mediana anual',    '@mediana{,}'),
])
p2.add_tools(hover2)

p2.xaxis.axis_label = 'Total de muertes acumuladas'
p2.yaxis.axis_label = 'País'
p2.xaxis.formatter.use_scientific = False
p2.title.text_font_size = '13px'
p2.axis.axis_label_text_font_style = 'bold'
p2.xgrid.grid_line_dash = [4, 4]
p2.ygrid.grid_line_color = None

show(p2)

In [19]:
# Unimos el dataset con las nuevas columnas creadas.
# los países con muertes en 0 no tendrán datos
df_export = df.merge(summary, on='Entity', how='left')

# Nombramos del archivo de salida
output_filename = 'doc_generado.csv'

# Exportamos el dataset 
df_export.to_csv(output_filename, index=False, encoding='utf-8-sig')